In [ ]:
import pandas as pd
import polars as pl
import pyarrow.dataset as ds

In [ ]:
ruta_parquet_tomtom = 'data/raw/tomtom_move_reports/segments_1_to_31_aug_2024.parquet'
df_tomtom_lazy = pl.scan_parquet(ruta_parquet_tomtom)

df_tomtom = df_tomtom_lazy.collect()
df_tomtom.head(20)

Usar filtros en caso de que no cuentes con la suficiente memoria fisica

In [ ]:
df_tomtom.estimated_size('gb')

In [ ]:
ruta_csv_sima = "data/processed/agosto2024.csv"
df_sima = pl.read_csv(ruta_csv_sima)
# filtrar a solo agosto
df_sima = df_sima.filter(
    pl.col("date").str.to_datetime(format="%Y-%m-%d %H:%M:%S").dt.month() == 8
)

In [ ]:
df_sima

## Calculo de candidatos


In [ ]:
df_tomtom = df_tomtom[['date', 'hour_start', 'station_name', 'segment_id', 'street_name', 'sample_size', 'travel_time_ratio', 'frc', 'avg_speed', 'distance_m']] # agrega columnas si lo requieres para calcular otra variable


# numero de muestras en la zona de la estacion
df_tomtom_suma_muestra = df_tomtom.group_by([
        'date',
        'hour_start',
        'station_name'
    ]).agg(
        pl.col('sample_size').sum().alias('total_sample_size_per_hour')
    ).sort(['station_name', 'date', 'hour_start'])

# avg_tti_street (de Ana Valverde)
df_tomtom_avg_tti_street = df_tomtom.filter(
    (pl.col("frc") <= 4)
    & (pl.col("sample_size") >= 10)
).group_by([
        'date',
        'hour_start',
        'station_name'
    ]).agg(
        pl.col("travel_time_ratio").mean().alias("avg_tti_street")
    ).sort(['station_name', 'date', 'hour_start'])


# Índice de Congestión Vehicular (de Rene)
import polars as pl

FRC_to_lanes = {
    0: 3,
    1: 2,
    2: 2,
    3: 2,
    4: 1,
    5: 1,
    6: 1,
    7: 1,
}

JAM_DENSITY_PER_LANE = 150  # por ejemplo

df_tomtom_vci_per_segment = (
    df_tomtom
    .with_columns(
        pl.col("frc").replace(FRC_to_lanes).alias("lanes")
    )
    .with_columns(
        pl.col("avg_speed").max().over("date").alias("free_flow_speed")
    )
    .with_columns(
        (JAM_DENSITY_PER_LANE * (1 - pl.col("avg_speed") / pl.col("free_flow_speed")))
        .alias("est_density_per_lane")
    )
    .with_columns(
        (pl.col("est_density_per_lane") * pl.col("lanes")).alias("est_density_total")
    )
    .with_columns(
        (pl.col("distance_m") / 1000).alias("distance_km")
    )
    .with_columns(
        (pl.col("est_density_total") * pl.col("distance_km"))
        .alias("estimated_vehicles_in_segment")
    )
    .with_columns(
        (pl.col("est_density_total") / (JAM_DENSITY_PER_LANE * pl.col("lanes")))
        .alias("congestion_ratio")
    )
)

df_tomtom_vci_mean = (df_tomtom_vci_per_segment
                      .group_by([
                        'date',
                        'hour_start',
                        'station_name'
                    ]).agg(
                        pl.col("congestion_ratio").mean().alias("congestion_ratio_mean"))
                        .sort(['station_name', 'date', 'hour_start']))

df_tomtom_vci_mean

## .

In [ ]:
df_sima = df_sima.with_columns(
    pl.col("date").str.to_datetime(format="%Y-%m-%d %H:%M:%S").alias("date")
)
df_sima  = df_sima.with_columns(
    pl.col("date").dt.date().cast(pl.Utf8).alias("date_only")
)
df_sima = df_sima[['date_only', 'Hora', 'Zona', 'CO', 'NOX', 'O3', 'PM2.5', 'PM10']]
df_sima

In [ ]:
df_sima_union_candidato = df_sima.join(
    df_tomtom_suma_muestra,
    left_on=['date_only', 'Hora', 'Zona'],
    right_on=['date', 'hour_start', 'station_name'],
    how='left'
)

df_sima_union_candidato = df_sima_union_candidato.join(
    df_tomtom_avg_tti_street,
    left_on=['date_only', 'Hora', 'Zona'],
    right_on=['date', 'hour_start', 'station_name'],
    how='left'
)

df_sima_union_candidato = df_sima_union_candidato.join(
    df_tomtom_vci_mean,
    left_on=['date_only', 'Hora', 'Zona'],
    right_on=['date', 'hour_start', 'station_name'],
    how='left'
)

df_sima_union_candidato

In [ ]:
# Remover fines de semana

df_sima_union_candidato = df_sima_union_candidato.filter(
    (pl.col('date_only').str.to_datetime(format="%Y-%m-%d").dt.weekday() != 6) &
    (pl.col('date_only').str.to_datetime(format="%Y-%m-%d").dt.weekday() != 7)
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def plot_candidate_v_contaminant(df ,candidate, contaminant):

    plt.figure(figsize=(10, 6))

    sns.regplot(
        x=candidate,
        y=contaminant,
        data=df.to_pandas(),
        scatter_kws={'alpha': 0.5},
        line_kws={'color': 'red', 'lw': 2}
    )

    mean_contaminant = df[contaminant].mean()

    plt.axhline(
        y=mean_contaminant,
        color='blue',
        linestyle='--',
        linewidth=2,
        label=f'Media de {contaminant}: {mean_contaminant:.2f}'
    )

    plt.title(f'Scatterplot de {candidate} vs {contaminant} en {df["Zona"][0]}')
    plt.xlabel(candidate)
    plt.ylabel(contaminant)
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
def plot_centros_candidate_v_contamintant(df, candidate, contaminant):
    df_sima_union_candidato_norte = df.filter(pl.col('Zona') == "NORTE")
    df_sima_union_candidato_sureste = df.filter(pl.col('Zona') == "SURESTE")
    df_sima_union_candidato_centro = df.filter(pl.col('Zona') == "CENTRO")
    plot_candidate_v_contaminant(df_sima_union_candidato_norte, candidate, contaminant)
    plot_candidate_v_contaminant(df_sima_union_candidato_sureste, candidate, contaminant)
    plot_candidate_v_contaminant(df_sima_union_candidato_centro, candidate, contaminant)

In [ ]:
plot_centros_candidate_v_contamintant(df_sima_union_candidato, 'total_sample_size_per_hour', 'O3')
plot_centros_candidate_v_contamintant(df_sima_union_candidato, 'avg_tti_street', 'O3')
plot_centros_candidate_v_contamintant(df_sima_union_candidato, 'congestion_ratio_mean', 'O3')